<a href="https://colab.research.google.com/github/jaw039/min-viable-eeg/blob/main/Full_Random_Channel_Control_Sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Full Random-Channel Control Sweep

In [ ]:
#Generate Random Subsets

import numpy as np
import pandas as pd
from itertools import combinations
from typing import List, Dict

class RandomChannelController:
    def __init__(self, all_channels: List[str], budgets: List[int],
                 n_subsets: int = 20, base_seed: int = 12345):
        self.all_channels = all_channels
        self.budgets = budgets
        self.n_subsets = n_subsets
        self.base_seed = base_seed
        self.subsets = {}

    def generate_subsets(self) -> Dict[int, List[Dict]]:
        """Generate unique random subsets for each budget."""
        for budget in self.budgets:
            subsets = []
            used_subsets = set()
            seed = self.base_seed

            while len(subsets) < self.n_subsets:
                np.random.seed(seed)
                selected = tuple(sorted(np.random.choice(
                    self.all_channels, budget, replace=False
                )))

                # Ensure uniqueness
                if selected not in used_subsets:
                    used_subsets.add(selected)
                    subsets.append({
                        'channels': list(selected),
                        'seed': seed,
                        'budget': budget
                    })
                seed += 1

            self.subsets[budget] = subsets
        return self.subsets

    def verify_subsets(self):
        """Verify all subsets are unique and valid."""
        for budget, subsets in self.subsets.items():
            # Check lengths
            assert all(len(s['channels']) == budget for s in subsets), \
                f"Budget {budget}: Incorrect subset length"

            # Check uniqueness
            channel_sets = [tuple(sorted(s['channels'])) for s in subsets]
            assert len(channel_sets) == len(set(channel_sets)), \
                f"Budget {budget}: Duplicate subsets found"

            # Check no invalid channels
            all_valid = all(ch in self.all_channels
                          for s in subsets for ch in s['channels'])
            assert all_valid, f"Budget {budget}: Invalid channel names"

        print(f"✓ All {sum(len(s) for s in self.subsets.values())} subsets verified")

In [ ]:
#Evaluate Random Subsets

class RandomExperimentRunner:
    def __init__(self, data_pipeline, model_class, training_config):
        self.data_pipeline = data_pipeline
        self.model_class = model_class
        self.training_config = training_config

    def run_random_sweep(self, controller: RandomChannelController) -> pd.DataFrame:
        """Run experiments for all random subsets."""
        results = []

        for budget, subsets in controller.subsets.items():
            print(f"\nBudget: {budget} channels")

            for idx, subset in enumerate(subsets):
                print(f"  Subset {idx+1}/{len(subsets)}: {subset['channels']}")

                # Set seed for reproducibility
                self._set_seed(subset['seed'])

                # Load data for this subset
                X_train, y_train = self.data_pipeline.load_data(
                    channels=subset['channels'], split='train'
                )
                X_val, y_val = self.data_pipeline.load_data(
                    channels=subset['channels'], split='val'
                )
                X_test, y_test = self.data_pipeline.load_data(
                    channels=subset['channels'], split='test'
                )

                # Train model
                model = self.model_class(
                    n_channels=budget,
                    **self.training_config
                )
                model = self._train_model(model, X_train, y_train, X_val, y_val)

                # Evaluate
                predictions = model.predict(X_test)
                kappa = self._calculate_kappa(y_test, predictions)
                accuracy = self._calculate_accuracy(y_test, predictions)

                # Store results
                results.append({
                    'budget': budget,
                    'method': 'random',
                    'subset_idx': idx,
                    'seed': subset['seed'],
                    'channels': subset['channels'],
                    'kappa': kappa,
                    'accuracy': accuracy,
                    'timestamp': pd.Timestamp.now()
                })

        return pd.DataFrame(results)

In [ ]:
#Validate Random Sampling

def validate_random_sampling(results_df: pd.DataFrame):
    """Validate that random sampling is working correctly."""

    # 1. Check all subsets have correct number of channels
    for budget in results_df['budget'].unique():
        mask = results_df['budget'] == budget
        n_subsets = len(results_df[mask])
        assert n_subsets == 20, f"Budget {budget}: Expected 20 subsets, got {n_subsets}"

        # Check all have correct channel count
        channel_counts = results_df[mask]['channels'].apply(len)
        assert (channel_counts == budget).all(), \
            f"Budget {budget}: Channel count mismatch"

    # 2. Check seeds are recorded
    assert not results_df['seed'].isnull().any(), "Missing seeds"
    assert len(results_df['seed'].unique()) == len(results_df), "Duplicate seeds"

    # 3. Check channel variety
    channel_frequency = {}
    for channels in results_df['channels']:
        for ch in channels:
            channel_frequency[ch] = channel_frequency.get(ch, 0) + 1

    # Check no channel appears too often (random selection should be uniform)
    n_subsets = len(results_df)
    n_channels = 64
    expected = n_subsets * 4 / n_channels  # for budget=4
    # Allow some variance

    print("✓ Random sampling validation passed")
    print(f"  Total experiments: {len(results_df)}")
    print(f"  Unique seeds: {len(results_df['seed'].unique())}")

In [ ]:
#Aggregate Random Results

def aggregate_random_results(results_df: pd.DataFrame) -> pd.DataFrame:
    """Compute statistics for random subsets at each budget."""

    agg = []
    for budget in sorted(results_df['budget'].unique()):
        mask = results_df['budget'] == budget
        budget_results = results_df[mask]

        agg.append({
            'budget': budget,
            'mean_kappa': budget_results['kappa'].mean(),
            'std_kappa': budget_results['kappa'].std(),
            'min_kappa': budget_results['kappa'].min(),
            'max_kappa': budget_results['kappa'].max(),
            'q25_kappa': budget_results['kappa'].quantile(0.25),
            'q75_kappa': budget_results['kappa'].quantile(0.75),
            'n_subsets': len(budget_results)
        })

    return pd.DataFrame(agg)

Analyze Selected vs. Random Performance

In [ ]:
#Compute Statistics

def analyze_selected_vs_random(selected_results, random_results, kappa_full):
    """Comprehensive analysis comparing selected and random performance."""

    analysis = {}

    for budget in sorted(selected_results['budget'].unique()):
        # Get results for this budget
        selected = selected_results[
            (selected_results['budget'] == budget) &
            (selected_results['method'] == 'selected')
        ]['kappa']

        random = random_results[
            (random_results['budget'] == budget) &
            (random_results['method'] == 'random')
        ]['kappa']

        # Basic statistics
        selected_mean = selected.mean()
        selected_std = selected.std()
        random_mean = random.mean()
        random_std = random.std()

        # Improvement
        improvement = selected_mean - random_mean
        relative_improvement = (improvement / random_mean) * 100

        # Statistical test (t-test)
        from scipy import stats
        t_stat, p_value = stats.ttest_ind(selected, random)

        # Cohen's d effect size
        pooled_std = np.sqrt((selected_std**2 + random_std**2) / 2)
        cohens_d = (selected_mean - random_mean) / pooled_std

        # Retention relative to full
        retention_selected = selected_mean / kappa_full
        retention_random = random_mean / kappa_full

        analysis[budget] = {
            'selected_kappa': f"{selected_mean:.3f} ± {selected_std:.3f}",
            'random_kappa': f"{random_mean:.3f} ± {random_std:.3f}",
            'improvement': f"{improvement:.3f} ({relative_improvement:.1f}%)",
            't_statistic': t_stat,
            'p_value': p_value,
            'significant': p_value < 0.05,
            'cohens_d': cohens_d,
            'retention_selected': retention_selected,
            'retention_random': retention_random
        }

    return pd.DataFrame(analysis).T

In [ ]:
#Interpret Results

def interpret_comparison(analysis_df):
    """Generate interpretation text for comparison results."""

    interpretation = []

    for budget, row in analysis_df.iterrows():
        text = f"\nBudget: {budget} channels\n"
        text += f"Selected: {row['selected_kappa']}\n"
        text += f"Random: {row['random_kappa']}\n"
        text += f"Improvement: {row['improvement']}\n"

        if row['significant']:
            text += f"✓ SIGNIFICANT (p={row['p_value']:.4f})\n"
            effect = 'large' if abs(row['cohens_d']) > 0.8 else \
                     'medium' if abs(row['cohens_d']) > 0.5 else 'small'
            text += f"  Effect size: {effect} (d={row['cohens_d']:.2f})\n"
        else:
            text += f"✗ NOT SIGNIFICANT (p={row['p_value']:.4f})\n"

        interpretation.append(text)

    return "\n".join(interpretation)


In [ ]:
#Create Visualization

import matplotlib.pyplot as plt
import seaborn as sns

def plot_selected_vs_random(selected_results, random_results, budgets):
    """Create comparison plot."""

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    # Plot 1: Bar plot with error bars
    ax = axes[0, 0]
    x = np.arange(len(budgets))
    width = 0.35

    selected_means = [selected_results[selected_results['budget']==b]['kappa'].mean()
                     for b in budgets]
    selected_stds = [selected_results[selected_results['budget']==b]['kappa'].std()
                     for b in budgets]
    random_means = [random_results[random_results['budget']==b]['kappa'].mean()
                    for b in budgets]
    random_stds = [random_results[random_results['budget']==b]['kappa'].std()
                   for b in budgets]

    ax.bar(x - width/2, selected_means, width, yerr=selected_stds,
           label='Selected', color='blue', alpha=0.7)
    ax.bar(x + width/2, random_means, width, yerr=random_stds,
           label='Random', color='red', alpha=0.7)
    ax.set_xlabel('Channel Budget')
    ax.set_ylabel('Cohen\'s κ')
    ax.set_title('Selected vs. Random Performance')
    ax.set_xticks(x)
    ax.set_xticklabels(budgets)
    ax.legend()

    # Plot 2: Box plot
    ax = axes[0, 1]
    selected_all = []
    random_all = []
    labels = []

    for b in budgets:
        selected_all.extend(selected_results[selected_results['budget']==b]['kappa'])
        random_all.extend(random_results[random_results['budget']==b]['kappa'])
        labels.extend([f'{b}\nSelected'] * len(selected_results[selected_results['budget']==b]))
        labels.extend([f'{b}\nRandom'] * len(random_results[random_results['budget']==b]))

    data = pd.DataFrame({
        'kappa': selected_all + random_all,
        'condition': labels
    })
    sns.boxplot(x='condition', y='kappa', data=data, ax=ax)
    ax.set_xlabel('Budget and Method')
    ax.set_ylabel('Cohen\'s κ')
    ax.set_title('Distribution Comparison')
    ax.tick_params(axis='x', rotation=45)

    # Plot 3: Improvement bar chart
    ax = axes[1, 0]
    improvements = [selected_means[i] - random_means[i] for i in range(len(budgets))]
    ax.bar(budgets, improvements, color='green', alpha=0.7)
    ax.axhline(y=0, color='black', linestyle='--')
    ax.set_xlabel('Channel Budget')
    ax.set_ylabel('Improvement (Selected - Random)')
    ax.set_title('Improvement of Selected vs. Random')

    # Plot 4: Retention plot
    ax = axes[1, 1]
    kappa_full = selected_results[selected_results['budget']==64]['kappa'].mean()
    selected_retention = [m / kappa_full for m in selected_means]
    random_retention = [m / kappa_full for m in random_means]

    ax.plot(budgets, selected_retention, 'b-o', label='Selected', linewidth=2)
    ax.plot(budgets, random_retention, 'r--s', label='Random', linewidth=2)
    ax.axhline(y=0.9, color='black', linestyle=':', label='90% Threshold')
    ax.set_xlabel('Channel Budget')
    ax.set_ylabel('Retention of Full Performance')
    ax.set_title('Performance Retention')
    ax.legend()

    plt.tight_layout()
    plt.savefig('selected_vs_random_analysis.png', dpi=300)
    plt.show()

Run/Review Repeated Experimental Seeds

In [ ]:
#Identify High-Variance Conditions

def identify_high_variance(results_df, threshold=0.05):
    """Find experimental conditions with high variance."""

    high_variance = []

    for (budget, method), group in results_df.groupby(['budget', 'method']):
        std_kappa = group['kappa'].std()
        mean_kappa = group['kappa'].mean()
        cv = std_kappa / mean_kappa  # Coefficient of variation

        if cv > threshold:
            high_variance.append({
                'budget': budget,
                'method': method,
                'mean_kappa': mean_kappa,
                'std_kappa': std_kappa,
                'cv': cv,
                'n_runs': len(group)
            })

    return pd.DataFrame(high_variance)

In [ ]:
#Re-run Critical Experiments

def repeat_experiments(high_variance_conditions, extra_seeds=10):
    """Re-run experiments with additional seeds."""

    additional_results = []

    for _, row in high_variance_conditions.iterrows():
        print(f"Re-running: Budget={row['budget']}, Method={row['method']}")

        for seed in range(extra_seeds):
            # Set seed
            np.random.seed(seed + 1000)  # Use different seed range

            # Re-run experiment
            result = run_single_experiment(
                budget=row['budget'],
                method=row['method'],
                seed=seed + 1000
            )
            additional_results.append(result)

    return pd.DataFrame(additional_results)

In [ ]:
#Verify Consistency

def verify_consistency(original_results, repeated_results):
    """Verify that repeated results are consistent with original."""

    consistency_check = []

    for (budget, method), group_orig in original_results.groupby(['budget', 'method']):
        group_rep = repeated_results[
            (repeated_results['budget'] == budget) &
            (repeated_results['method'] == method)
        ]

        if len(group_rep) == 0:
            continue

        # Compare means
        orig_mean = group_orig['kappa'].mean()
        rep_mean = group_rep['kappa'].mean()
        mean_diff = abs(orig_mean - rep_mean)

        # Statistical test
        from scipy import stats
        t_stat, p_value = stats.ttest_ind(group_orig['kappa'], group_rep['kappa'])

        consistency_check.append({
            'budget': budget,
            'method': method,
            'original_mean': orig_mean,
            'repeated_mean': rep_mean,
            'mean_difference': mean_diff,
            'p_value': p_value,
            'consistent': p_value > 0.05
        })

    return pd.DataFrame(consistency_check)